# Modèle de Détection d'Anomalies

---

## Objectif de ce notebook

Identifier automatiquement les observations anormales dans les données de stock :  
fuites de produit, erreurs de saisie, variations inexplicables, détournements éventuels.

**Type d'apprentissage :** Non supervisé — le modèle apprend ce qui est normal  
et signale ce qui s'écarte de cette norme, **sans** avoir besoin d'exemples d'anomalies étiquetés  
au préalable. La colonne `anomalie_detectee` sert **uniquement à l'évaluation**, pas à l'entraînement.

**Algorithmes comparés :** Isolation Forest vs Z-score  
**Table utilisée :** `stocks_journaliers` (321 464 observations)  
**Proportion d'anomalies dans le dataset :** 1,02% (3 294 sur 321 464)

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Dossiers racine pour les figures et modèles
FIGURES_DIR = os.path.join('..', 'figures')
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

if not hasattr(plt.savefig, '_is_patched'):
    _original_savefig = plt.savefig
    def savefig_root(path, *args, **kwargs):
        if isinstance(path, str) and path.startswith('figures/'):
            path = os.path.join('..', path)
        return _original_savefig(path, *args, **kwargs)
    savefig_root._is_patched = True
    plt.savefig = savefig_root

print('Imports OK ✅')

## 1. Chargement et Préparation des Données

In [ ]:
df = pd.read_csv('../data/stocks_journaliers.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset : {df.shape[0]:,} observations × {df.shape[1]} colonnes")
print(f"Anomalies : {df['anomalie_detectee'].sum():,} ({df['anomalie_detectee'].mean()*100:.2f}%)")
print(f"Normales  : {(~df['anomalie_detectee'].astype(bool)).sum():,}")

## 2. Sélection et Justification des Variables d'Entrée

Le choix des variables est guidé par les observations de l'EDA (notebook 01) :
- `stock_fin_jour` et `taux_remplissage_pct` sont les plus discriminants (écart de -20% et -18,9% entre normales et anomalies)
- `entrees` et `sorties` ont peu de corrélation avec les anomalies → inclus pour capturer des patterns opérationnels
- `valeur_stock` est **exclu** car redondant avec `stock_fin_jour × prix_unitaire`
- `prix_wti_usd_baril` et `prix_unitaire` sont **exclus** car non liés aux anomalies de stock physique
- On ajoute des variables temporelles pour capturer les patterns horaires/saisonniers

In [ ]:
# Ajout de variables temporelles
df['mois']       = df['date'].dt.month
df['jour_sem']   = df['date'].dt.dayofweek
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
df['trimestre']  = df['date'].dt.quarter

# Variables d'entrée retenues
FEATURES = [
    'stock_fin_jour',        # Variable la plus discriminante (écart -20%)
    'taux_remplissage_pct',  # Variable la plus discriminante (écart -18,9%)
    'entrees',               # Flux entrant
    'sorties',               # Flux sortant
    'stock_debut_jour',      # Niveau de départ
    'mois',                  # Saisonnalité mensuelle
    'jour_sem',              # Saisonnalité hebdomadaire
    'is_weekend',            # Pattern weekend
]

# Séparation features / cible (la cible sert UNIQUEMENT à l'évaluation)
X = df[FEATURES].copy()
y_true = df['anomalie_detectee'].astype(int)  # 0=Normal, 1=Anomalie

print(f"Features sélectionnées : {len(FEATURES)}")
for f in FEATURES:
    print(f"  - {f}")
print(f"\nValeurs manquantes : {X.isnull().sum().sum()}")

## 3. Normalisation

Isolation Forest et Z-score sont tous deux sensibles aux échelles des variables.  
On applique une normalisation **StandardScaler** (moyenne=0, std=1) sur toutes les features.  
Le scaler sera sauvegardé avec le modèle — il est indispensable pour l'API.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)

print("Normalisation appliquée ✅")
print(f"\nVérification (moyennes après normalisation — doivent être ≈ 0) :")
print(X_scaled_df.mean().round(4))
print(f"\nVérification (std après normalisation — doivent être ≈ 1) :")
print(X_scaled_df.std().round(4))

## 4. Algorithme 1 — Isolation Forest

Isolation Forest détecte les anomalies en isolant les observations dans des arbres de décision aléatoires.  
Une observation est anormale si elle est facilement isolée (peu de coupures nécessaires).  

**Paramètre `contamination` :** fixé à **0.01** (1%) — justifié par la proportion observée dans le dataset (1,02%).  
**Paramètre `n_estimators` :** 200 arbres — bon compromis précision/performance.

In [ ]:
# Entraînement Isolation Forest
model_if = IsolationForest(
    n_estimators=200,
    contamination=0.01,  # 1% justifié par l'EDA
    random_state=42,
    n_jobs=-1
)
model_if.fit(X_scaled)
print("Isolation Forest entraîné ✅")

# Prédictions : -1=Anomalie, 1=Normal → conversion en 0/1
pred_if_raw = model_if.predict(X_scaled)
pred_if = np.where(pred_if_raw == -1, 1, 0)  # 1=Anomalie, 0=Normal

# Scores d'anomalie (plus négatif = plus anormal)
scores_if = model_if.score_samples(X_scaled)
# Normalisation du score entre 0 et 1 (1 = très anormal)
scores_if_norm = 1 - (scores_if - scores_if.min()) / (scores_if.max() - scores_if.min())

print(f"\nAnomalies détectées : {pred_if.sum():,} ({pred_if.mean()*100:.2f}%)")
print(f"Score d'anomalie moyen : {scores_if_norm.mean():.3f}")
print(f"Score d'anomalie max   : {scores_if_norm.max():.3f}")

In [ ]:
# Métriques Isolation Forest
print("=== Métriques Isolation Forest ===")
print(classification_report(y_true, pred_if,
                             target_names=['Normal', 'Anomalie']))

prec_if = precision_score(y_true, pred_if)
rec_if  = recall_score(y_true, pred_if)
f1_if   = f1_score(y_true, pred_if)

print(f"Précision : {prec_if:.3f}")
print(f"Rappel    : {rec_if:.3f}")
print(f"F1-score  : {f1_if:.3f}")

In [ ]:
# Visualisation des anomalies sur la courbe temporelle
# Gasoil, Dépôt Central Lomé
mask = (df['produit_id'] == 'PRD003') & (df['depot_id'] == 'D001')
df_ex = df[mask].copy()
df_ex['pred_if'] = pred_if[mask.values]
df_ex['score_if'] = scores_if_norm[mask.values]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_ex['date'], df_ex['stock_fin_jour'],
        color='steelblue', linewidth=0.7, label='Stock normal')
anomalies_if = df_ex[df_ex['pred_if'] == 1]
ax.scatter(anomalies_if['date'], anomalies_if['stock_fin_jour'],
           color='red', s=30, zorder=5, label=f'Anomalie IF ({len(anomalies_if)})')
ax.set_title('Isolation Forest — Anomalies détectées sur Gasoil · Dépôt Central Lomé')
ax.set_xlabel('Date')
ax.set_ylabel('Stock fin de jour')
ax.legend()
plt.tight_layout()
plt.savefig('figures/01_if_anomalies_courbe.png', dpi=150)
plt.show()

In [ ]:
# Distribution des scores d'anomalie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme des scores
axes[0].hist(scores_if_norm[y_true == 0], bins=50,
             alpha=0.7, color='steelblue', label='Normales', density=True)
axes[0].hist(scores_if_norm[y_true == 1], bins=50,
             alpha=0.7, color='crimson', label='Anomalies réelles', density=True)
axes[0].set_title('Distribution des scores d\'anomalie — Isolation Forest')
axes[0].set_xlabel('Score (0=normal, 1=très anormal)')
axes[0].set_ylabel('Densité')
axes[0].legend()

# Matrice de confusion IF
cm_if = confusion_matrix(y_true, pred_if)
sns.heatmap(cm_if, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalie'],
            yticklabels=['Normal', 'Anomalie'], ax=axes[1])
axes[1].set_title('Matrice de confusion — Isolation Forest')
axes[1].set_xlabel('Prédit')
axes[1].set_ylabel('Réel')

plt.tight_layout()
plt.savefig('figures/02_if_scores_confusion.png', dpi=150)
plt.show()

## 5. Algorithme 2 — Z-score

Le Z-score mesure l'écart d'une observation par rapport à la moyenne en unités d'écart-type.  
Une observation avec |Z| > seuil est considérée anormale.  

**Seuil retenu :** |Z| > **2.5** (au lieu du classique 3.0).  
Justification : avec seulement 1% d'anomalies et des distributions asymétriques (stocks),  
un seuil à 3.0 serait trop conservateur et raterait trop d'anomalies réelles.  
Le seuil à 2.5 offre un meilleur rappel sur ce dataset.

In [ ]:
# Calcul du Z-score sur les variables clés
SEUIL_Z = 2.5
cols_zscore = ['stock_fin_jour', 'taux_remplissage_pct', 'sorties', 'entrees']

z_scores = np.abs(stats.zscore(df[cols_zscore].fillna(0)))
z_scores_df = pd.DataFrame(z_scores, columns=cols_zscore)

# Une observation est anormale si AU MOINS UNE variable dépasse le seuil
pred_z = (z_scores_df > SEUIL_Z).any(axis=1).astype(int)

# Score Z-score : max des Z-scores normalisé
scores_z = z_scores_df.max(axis=1)
scores_z_norm = (scores_z / scores_z.max()).clip(0, 1)

print(f"Z-score calculé (seuil = {SEUIL_Z}) ✅")
print(f"Anomalies détectées : {pred_z.sum():,} ({pred_z.mean()*100:.2f}%)")
print(f"\nZ-scores max par variable :")
for c in cols_zscore:
    print(f"  {c:<30} : max Z = {z_scores_df[c].max():.2f}")

In [ ]:
# Métriques Z-score
print("=== Métriques Z-score ===")
print(classification_report(y_true, pred_z,
                             target_names=['Normal', 'Anomalie']))

prec_z = precision_score(y_true, pred_z)
rec_z  = recall_score(y_true, pred_z)
f1_z   = f1_score(y_true, pred_z)

print(f"Précision : {prec_z:.3f}")
print(f"Rappel    : {rec_z:.3f}")
print(f"F1-score  : {f1_z:.3f}")

# Matrice de confusion Z-score
cm_z = confusion_matrix(y_true, pred_z)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_z, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal', 'Anomalie'],
            yticklabels=['Normal', 'Anomalie'], ax=ax)
ax.set_title('Matrice de confusion — Z-score')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.savefig('figures/03_zscore_confusion.png', dpi=150)
plt.show()

## 6. Comparaison et Sélection du Meilleur Modèle

In [ ]:
# Tableau comparatif
resultats = pd.DataFrame({
    'Modèle'    : ['Isolation Forest', 'Z-score'],
    'Précision' : [prec_if, prec_z],
    'Rappel'    : [rec_if,  rec_z],
    'F1-score'  : [f1_if,   f1_z],
    'Anomalies détectées' : [pred_if.sum(), pred_z.sum()]
})
print(resultats.to_string(index=False))

In [ ]:
# Barplot de comparaison des métriques
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metriques = ['Précision', 'Rappel', 'F1-score']
vals_if = [prec_if, rec_if, f1_if]
vals_z  = [prec_z,  rec_z,  f1_z]

for i, (met, vif, vz) in enumerate(zip(metriques, vals_if, vals_z)):
    bars = axes[i].bar(['Isolation Forest', 'Z-score'], [vif, vz],
                        color=['steelblue', 'darkorange'], edgecolor='white', width=0.5)
    for bar, val in zip(bars, [vif, vz]):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.005,
                     f'{val:.3f}', ha='center', fontsize=11)
    axes[i].set_title(met)
    axes[i].set_ylim(0, 1.1)
    axes[i].set_ylabel(met)

fig.suptitle('Comparaison Isolation Forest vs Z-score', fontsize=13)
plt.tight_layout()
plt.savefig('figures/04_comparaison_metriques.png', dpi=150)
plt.show()

In [ ]:
# Anomalies communes aux deux méthodes vs spécifiques
communs = ((pred_if == 1) & (pred_z == 1)).sum()
only_if = ((pred_if == 1) & (pred_z == 0)).sum()
only_z  = ((pred_if == 0) & (pred_z == 1)).sum()

print(f"Anomalies détectées par les DEUX méthodes : {communs:,}")
print(f"Anomalies détectées uniquement par IF      : {only_if:,}")
print(f"Anomalies détectées uniquement par Z-score : {only_z:,}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Communes\n(IF + Z-score)', 'IF seulement', 'Z-score seulement'],
               [communs, only_if, only_z],
               color=['#1E8E5A', 'steelblue', 'darkorange'], edgecolor='white')
for bar, val in zip(bars, [communs, only_if, only_z]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5, str(val), ha='center', fontsize=11)
ax.set_title('Anomalies communes vs spécifiques à chaque méthode')
ax.set_ylabel("Nombre d'anomalies")
plt.tight_layout()
plt.savefig('figures/05_anomalies_communes.png', dpi=150)
plt.show()

**📝 Réflexion — Précision vs Rappel dans ce contexte métier :**

> Dans le contexte de la détection d'anomalies de stock pétrolier, la question est :  
> **est-il plus grave de manquer une vraie anomalie ou de déclencher une fausse alerte ?**
>
> **Manquer une vraie anomalie (faux négatif)** signifie ne pas détecter une fuite, un détournement ou une erreur de gestion — conséquences potentiellement très graves : pertes financières, risques environnementaux, non-conformité réglementaire.
>
> **Déclencher une fausse alerte (faux positif)** signifie mobiliser inutilement les équipes pour une vérification — coût opérationnel faible, simple perte de temps.
>
> **Conclusion : le RAPPEL est la métrique prioritaire dans ce contexte.** On préfère déclencher quelques fausses alertes plutôt que de rater une vraie anomalie. Le modèle avec le meilleur rappel sera donc privilégié, à condition que la précision reste acceptable.

**📝 Sélection et Justification :**

> Le modèle retenu est **Isolation Forest** pour les raisons suivantes :
> - Il produit un **score d'anomalie continu** (entre 0 et 1) permettant de prioriser les alertes par niveau de suspicion — le Z-score produit uniquement un binaire.
> - Il est plus robuste aux distributions non gaussiennes, fréquentes dans les données de stock (distributions asymétriques avec queues épaisses).
> - Sa performance F1-score est meilleure ou équivalente au Z-score selon les données.
> - Il est nativement intégrable dans une pipeline scikit-learn pour l'API.
>
> Le Z-score reste utile comme **méthode de vérification complémentaire** — les anomalies détectées par les **deux méthodes simultanément** ont le niveau de confiance le plus élevé et seront marquées comme prioritaires dans le dashboard.

> ⚠️ **Point de vigilance :** le paramètre `contamination=0.01` a été calé sur le taux réel d'anomalies observé dans la colonne cible `anomalie_detectee`. Ce taux ne sera **pas connu à l'avance en production** — le modèle n'est donc pas totalement "non supervisé" tel qu'entraîné ici. À reconsidérer : soit fixer `contamination` sur une estimation métier indépendante des labels, soit basculer sur `contamination='auto'` et valider le résultat avec les équipes terrain.

## 7. Sauvegarde du Modèle et du Scaler

In [ ]:
# Sauvegarder le modèle Isolation Forest
with open(os.path.join(MODELS_DIR, 'model_anomalies.pkl'), 'wb') as f:
    pickle.dump(model_if, f)

# Sauvegarder le scaler (INDISPENSABLE pour l'API)
with open(os.path.join(MODELS_DIR, 'scaler_anomalies.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"Modèle sauvegardé   : {os.path.join(MODELS_DIR, 'model_anomalies.pkl')} ✅")
print(f"Scaler sauvegardé   : {os.path.join(MODELS_DIR, 'scaler_anomalies.pkl')} ✅")

## 8. Conclusions

---

### 🔍 Résultats clés

> 1. **Isolation Forest** avec `contamination=0.01` (justifié par la proportion réelle de 1,02%) produit des résultats cohérents avec la réalité des données.
> 2. Le **rappel est la métrique prioritaire** dans ce contexte — mieux vaut une fausse alerte qu'une vraie anomalie manquée.
> 3. Les anomalies détectées par **les deux méthodes simultanément** ont le niveau de confiance le plus élevé — à traiter en priorité.
> 4. Le **score d'anomalie continu** d'Isolation Forest (0 à 1) permet de prioriser les alertes sur le dashboard — bien supérieur au binaire du Z-score.
> 5. Les deux fichiers à transmettre à SEGNEDJI : `model_anomalies.pkl` ET `scaler_anomalies.pkl`.

---

### ✅ Décisions

> - **Modèle retenu** : Isolation Forest (`contamination=0.01`, `n_estimators=200`)
> - **Fichiers sauvegardés** : `models/model_anomalies.pkl` + `models/scaler_anomalies.pkl`
> - **Seuil d'alerte dashboard** : score > 0.7 → alerte modérée · score > 0.85 → alerte critique
> - **Stratégie complémentaire** : anomalies communes IF + Z-score → badge "Haute confiance"

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_if_anomalies_courbe.png` | Anomalies IF sur la courbe temporelle |
| `02_if_scores_confusion.png` | Distribution des scores + matrice de confusion IF |
| `03_zscore_confusion.png` | Matrice de confusion Z-score |
| `04_comparaison_metriques.png` | Comparaison Précision / Rappel / F1 |
| `05_anomalies_communes.png` | Anomalies communes vs spécifiques |